# Step 0.4 - OpenPhenom image embeddings (384 and 1,920 dimensions)

**What this notebook does.** It attaches compound metadata to OpenPhenom embeddings of JUMP Cell
Painting wells and saves them as Parquet, for two embedding sizes.

**Background.** OpenPhenom is an open-source foundation model for cell microscopy images released by
Recursion. Like DINO (Step 0.3) it is a vision transformer trained without labels, in this case as a
*masked autoencoder*: it learns by reconstructing hidden patches of images. Embeddings were computed
for single cells and then averaged per well ("aggregated profiles"), at two sizes: 384 and 1,920
dimensions.

**Inputs.** `aggregated_profiles_openphenom_single_cell_384.csv.gz`,
`aggregated_profiles_openphenom_single_cell_1920.csv.gz` and the JUMP compound metadata table.

**Outputs.** `openphenom_384.parquet` and `openphenom_1920.parquet`: one row per well, indexed by InChIKey.

**Where to run it.** Written for Google Colab / Colab Enterprise: it uses the `gsutil` command-line tool and
caches immutable downloads under the repository-level `data/raw/` directory and retains assembled local
outputs under this stage's `results/` directory. You need read/write access to the
`gs://calico-reference-embeddings` Cloud Storage bucket for the remote sources and uploads.

## Setup

All libraries used in this notebook are imported in the next cell. Locally they are installed by `uv sync`
(see the main README); on Google Colab, first run
`!pip install git+https://github.com/axiomcura/ModelBenchmarking.git`, which installs the same dependencies.

In [1]:
import pandas as pd
from IPython.display import display

from model_benchmarking.paths import cache_gcs_file, notebook_artifact_dirs

RAW_DATA_DIR, RESULTS_DIR = notebook_artifact_dirs("0.download_and_assemble", "openphenom")
METADATA_RAW_DIR, _ = notebook_artifact_dirs("0.download_and_assemble", "compound_metadata")


### Shared constants
Identifiers used to label DMSO control wells.

In [2]:
DMSO_INCHIKEY = "IAZDPXIOMUYVGZ-UHFFFAOYSA-N"
DMSO_SMILES = "CS(C)=O"

## Part A - 384-dimensional embeddings

### A.1 Configuration

In [3]:
openphenom_gcs = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/single-cells_384_features/aggregated_profiles_openphenom_single_cell_384.csv.gz'
metadata_gcs = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/JUMP_CP_compound_metadata_with_SMILES_standard.csv'
openphenom_csv = cache_gcs_file(openphenom_gcs, RAW_DATA_DIR / 'aggregated_profiles_openphenom_single_cell_384.csv.gz')
metadata_csv = cache_gcs_file(metadata_gcs, METADATA_RAW_DIR / 'JUMP_CP_compound_metadata_with_SMILES_standard.csv')

out_parquet = RESULTS_DIR / 'openphenom_384.parquet'
gcs_out_parquet = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_384.parquet'


### A.2 Load the embeddings and the metadata

In [4]:
print("📂 Loading OpenPhenom Embeddings (Decompressing GZIP)...")
df_emb = pd.read_csv(openphenom_csv, compression='gzip')

print("📂 Loading Metadata...")
df_meta = pd.read_csv(metadata_csv)

print("\n📊 [DIAGNOSTIC] PRE-JOIN COLUMN COUNTS:")
print(f"   -> OpenPhenom: {df_emb.shape[1]} columns")
print(f"   -> Metadata:   {df_meta.shape[1]} columns")

### A.3 Separate embedding and metadata columns
Embedding columns are named `0, 1, 2, …` or `emb_…`; everything else is metadata, which is moved to the left.

In [5]:
# Find the exact 384 embedding columns (Assuming they are named '0', '1', '2'...)
emb_cols = [c for c in df_emb.columns if str(c).isdigit() or str(c).startswith('emb_')]
# Everything else in OpenPhenom is metadata
op_meta_cols = [c for c in df_emb.columns if c not in emb_cols]

print(f"\n⚙️ Shifting {len(op_meta_cols)} metadata columns to the front of OpenPhenom...")
df_emb = df_emb[op_meta_cols + emb_cols]

### A.4 Find the shared identifier column

In [6]:
possible_join_keys = ['InChIKey', 'inchikey', 'Metadata_JCP2022', 'perturbation_ID', 'perturbation_id']

JOIN_COL = next((c for c in possible_join_keys if c in df_emb.columns and c in df_meta.columns), None)
if not JOIN_COL:
    raise KeyError(f"❌ Could not find a shared identifier to join on! \nEmb Meta: {op_meta_cols}\nAnno Meta: {list(df_meta.columns[:10])}")

print(f"   -> Detected Join Key: '{JOIN_COL}'")

# Deduplicate metadata on the join key to prevent multiplying rows
df_meta = df_meta.drop_duplicates(subset=[JOIN_COL])

### A.5 Attach metadata
A *left join* keeps every well. When both tables contain a column with the same name, the official JUMP metadata wins and the OpenPhenom value only fills gaps.

In [7]:
print("\n🔗 Executing LEFT JOIN (Preserving all columns)...")

# We use suffixes so we can explicitly see overlapping columns (e.g. 'smiles_OP' and 'smiles_META')
df_joined = pd.merge(df_emb, df_meta, on=JOIN_COL, how='left', suffixes=('_OP', '_META'))

# Resolve overlapping columns (Prioritizing the official Metadata file, fallback to OpenPhenom)
overlap_bases = [c.replace('_META', '') for c in df_joined.columns if c.endswith('_META')]

for col in overlap_bases:
    col_op = f"{col}_OP"
    col_meta = f"{col}_META"

    # Combine them into a single clean column
    df_joined[col] = df_joined[col_meta].combine_first(df_joined[col_op])
    # Drop the messy suffix columns
    df_joined.drop(columns=[col_op, col_meta], inplace=True)

### A.6 Label DMSO control wells and index by InChIKey
DMSO wells (solvent only) have no entry in the compound table. They are recognized by their control-type label (`negcon`/`control`) and given DMSO's InChIKey and SMILES, so they can act as the "untreated" reference in Step 1.1. Wells that still lack an InChIKey are dropped.

In [8]:
print("\n🛡️ Rescuing DMSO controls...")

# Identify the InChIKey column for indexing
INDEX_COL = next((c for c in df_joined.columns if 'inchikey' in c.lower()), None)
if not INDEX_COL:
    raise KeyError("❌ Could not find an InChIKey column!")

CONTROL_COL = next((c for c in ['control_type', 'Metadata_control_type', 'pert_type'] if c in df_joined.columns), None)

if CONTROL_COL:
    is_dmso = df_joined[CONTROL_COL].astype(str).str.strip().str.lower().isin(['negcon', 'control'])
    df_joined.loc[is_dmso, INDEX_COL] = DMSO_INCHIKEY

    SMILES_COL = next((c for c in df_joined.columns if 'smiles' in c.lower()), None)
    if SMILES_COL:
        df_joined.loc[is_dmso, SMILES_COL] = DMSO_SMILES

# Clean string and explicitly drop rows that STILL don't have an InChIKey
df_joined = df_joined.dropna(subset=[INDEX_COL])
df_joined[INDEX_COL] = df_joined[INDEX_COL].astype(str).str.strip()

print(f"⚙️ Setting '{INDEX_COL}' as the primary index...")
df_joined.set_index(INDEX_COL, inplace=True)
df_joined.index.name = 'InChIKey'

### A.7 Save
Metadata columns are stored as text, followed by the embedding columns.

In [9]:
# Ensure metadata is on the left and embeddings are on the right
final_meta_cols = [c for c in df_joined.columns if c not in emb_cols]
df_joined = df_joined[final_meta_cols + emb_cols]

# Force all metadata columns to string to prevent PyArrow crashes
for col in final_meta_cols:
    df_joined[col] = df_joined[col].astype(str)

print("\n📊 [DIAGNOSTIC] POST-JOIN COLUMN COUNTS:")
print(f"   -> Final Metadata Columns:  {len(final_meta_cols)}")
print(f"   -> Final Embedding Columns: {len(emb_cols)}")
print(f"   -> TOTAL COLUMNS:           {df_joined.shape[1]}")

print(f"\n💾 Saving highly-optimized Parquet to GCS...")
df_joined.to_parquet(out_parquet, compression='snappy')
df_joined.to_parquet(gcs_out_parquet, compression='snappy')

print(f"🎉 SUCCESS! Files saved to:\n   Local: {out_parquet}\n   GCS:   {gcs_out_parquet}")
display(df_joined.head(5))

## Part B - 1,920-dimensional embeddings

The identical procedure applied to the 1,920-dimensional file. (The code comment "Find the exact 384
embedding columns" is inherited from Part A; columns are selected by name, so all 1,920 are captured.)

### B.1 Configuration

In [10]:
openphenom_gcs = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/single-cells_1920_features/aggregated_profiles_openphenom_single_cell_1920.csv.gz'
metadata_gcs = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/JUMP_CP_compound_metadata_with_SMILES_standard.csv'
openphenom_csv = cache_gcs_file(openphenom_gcs, RAW_DATA_DIR / 'aggregated_profiles_openphenom_single_cell_1920.csv.gz')
metadata_csv = cache_gcs_file(metadata_gcs, METADATA_RAW_DIR / 'JUMP_CP_compound_metadata_with_SMILES_standard.csv')

out_parquet = RESULTS_DIR / 'openphenom_1920.parquet'
gcs_out_parquet = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_1920.parquet'


### B.2 Load the embeddings and the metadata

In [11]:
print("📂 Loading OpenPhenom Embeddings (Decompressing GZIP)...")
df_emb = pd.read_csv(openphenom_csv, compression='gzip')

print("📂 Loading Metadata...")
df_meta = pd.read_csv(metadata_csv)

print("\n📊 [DIAGNOSTIC] PRE-JOIN COLUMN COUNTS:")
print(f"   -> OpenPhenom: {df_emb.shape[1]} columns")
print(f"   -> Metadata:   {df_meta.shape[1]} columns")

### B.3 Separate embedding and metadata columns
Embedding columns are named `0, 1, 2, …` or `emb_…`; everything else is metadata, which is moved to the left.

In [12]:
# Find the exact 384 embedding columns (Assuming they are named '0', '1', '2'...)
emb_cols = [c for c in df_emb.columns if str(c).isdigit() or str(c).startswith('emb_')]
# Everything else in OpenPhenom is metadata
op_meta_cols = [c for c in df_emb.columns if c not in emb_cols]

print(f"\n⚙️ Shifting {len(op_meta_cols)} metadata columns to the front of OpenPhenom...")
df_emb = df_emb[op_meta_cols + emb_cols]

### B.4 Find the shared identifier column

In [13]:
possible_join_keys = ['InChIKey', 'inchikey', 'Metadata_JCP2022', 'perturbation_ID', 'perturbation_id']

JOIN_COL = next((c for c in possible_join_keys if c in df_emb.columns and c in df_meta.columns), None)
if not JOIN_COL:
    raise KeyError(f"❌ Could not find a shared identifier to join on! \nEmb Meta: {op_meta_cols}\nAnno Meta: {list(df_meta.columns[:10])}")

print(f"   -> Detected Join Key: '{JOIN_COL}'")

# Deduplicate metadata on the join key to prevent multiplying rows
df_meta = df_meta.drop_duplicates(subset=[JOIN_COL])

### B.5 Attach metadata
A *left join* keeps every well. When both tables contain a column with the same name, the official JUMP metadata wins and the OpenPhenom value only fills gaps.

In [14]:
print("\n🔗 Executing LEFT JOIN (Preserving all columns)...")

# We use suffixes so we can explicitly see overlapping columns (e.g. 'smiles_OP' and 'smiles_META')
df_joined = pd.merge(df_emb, df_meta, on=JOIN_COL, how='left', suffixes=('_OP', '_META'))

# Resolve overlapping columns (Prioritizing the official Metadata file, fallback to OpenPhenom)
overlap_bases = [c.replace('_META', '') for c in df_joined.columns if c.endswith('_META')]

for col in overlap_bases:
    col_op = f"{col}_OP"
    col_meta = f"{col}_META"

    # Combine them into a single clean column
    df_joined[col] = df_joined[col_meta].combine_first(df_joined[col_op])
    # Drop the messy suffix columns
    df_joined.drop(columns=[col_op, col_meta], inplace=True)

### B.6 Label DMSO control wells and index by InChIKey
DMSO wells (solvent only) have no entry in the compound table. They are recognized by their control-type label (`negcon`/`control`) and given DMSO's InChIKey and SMILES, so they can act as the "untreated" reference in Step 1.1. Wells that still lack an InChIKey are dropped.

In [15]:
print("\n🛡️ Rescuing DMSO controls...")

# Identify the InChIKey column for indexing
INDEX_COL = next((c for c in df_joined.columns if 'inchikey' in c.lower()), None)
if not INDEX_COL:
    raise KeyError("❌ Could not find an InChIKey column!")

CONTROL_COL = next((c for c in ['control_type', 'Metadata_control_type', 'pert_type'] if c in df_joined.columns), None)

if CONTROL_COL:
    is_dmso = df_joined[CONTROL_COL].astype(str).str.strip().str.lower().isin(['negcon', 'control'])
    df_joined.loc[is_dmso, INDEX_COL] = DMSO_INCHIKEY

    SMILES_COL = next((c for c in df_joined.columns if 'smiles' in c.lower()), None)
    if SMILES_COL:
        df_joined.loc[is_dmso, SMILES_COL] = DMSO_SMILES

# Clean string and explicitly drop rows that STILL don't have an InChIKey
df_joined = df_joined.dropna(subset=[INDEX_COL])
df_joined[INDEX_COL] = df_joined[INDEX_COL].astype(str).str.strip()

print(f"⚙️ Setting '{INDEX_COL}' as the primary index...")
df_joined.set_index(INDEX_COL, inplace=True)
df_joined.index.name = 'InChIKey'

### B.7 Save
Metadata columns are stored as text, followed by the embedding columns.

In [16]:
# Ensure metadata is on the left and embeddings are on the right
final_meta_cols = [c for c in df_joined.columns if c not in emb_cols]
df_joined = df_joined[final_meta_cols + emb_cols]

# Force all metadata columns to string to prevent PyArrow crashes
for col in final_meta_cols:
    df_joined[col] = df_joined[col].astype(str)

print("\n📊 [DIAGNOSTIC] POST-JOIN COLUMN COUNTS:")
print(f"   -> Final Metadata Columns:  {len(final_meta_cols)}")
print(f"   -> Final Embedding Columns: {len(emb_cols)}")
print(f"   -> TOTAL COLUMNS:           {df_joined.shape[1]}")

print(f"\n💾 Saving highly-optimized Parquet to GCS...")
df_joined.to_parquet(out_parquet, compression='snappy')
df_joined.to_parquet(gcs_out_parquet, compression='snappy')

print(f"🎉 SUCCESS! Files saved to:\n   Local: {out_parquet}\n   GCS:   {gcs_out_parquet}")
display(df_joined.head(5))